# Compute start-specific two-reference drift diagnostics

For every start year and monthly lead, compute the ensemble mean, departures from observations and the E3SM model attractor, changes from the `L=1` initialization-month baseline, absolute-distance changes, and tolerance-aware drift regimes. `Y` is retained in every primary output.

In [ ]:
from workflows.diagnostics.two_reference_drift import (
    expand_requests, reference_output_path, run_job,
)

REQUESTS = [{
    'sources': ['Reanalysis', 'JRA55_FOSIRL'],
    'component': 'atm',
    'variables': ['TREFHT'],
    'init_months': [5],
    'regrid': True,
}]
DISTANCE_TOLERANCE = 1.0e-6  # degC for the TREFHT pilot; revisit scientifically
CLIMATOLOGY_YEARS = (1981, 2010)
ANALYSIS_YEARS = (1981, 2010)
FORCE = False

In [ ]:
outputs = {}
for job in expand_requests(REQUESTS):
    references = reference_output_path(job)
    if not references.is_file():
        raise FileNotFoundError(f'Run 0_run_drift_references.ipynb first: {references}')
    path = run_job(
        job, prepared_reference_path=references,
        climatology_years=CLIMATOLOGY_YEARS, analysis_years=ANALYSIS_YEARS,
        distance_tolerance=DISTANCE_TOLERANCE, force=FORCE,
    )
    outputs[(job['source'], job['init_month'], job['variable'])] = path
    print(f'Ready: {path}')
outputs

## Validate saved products and paired coordinates

The formulas are validated before writing. This cell verifies the reusable files and confirms that Reanalysis and FOSIRL have identical `Y`, `L`, and valid-time coordinates before a later paired comparison.

In [ ]:
import numpy as np
import xarray as xr
from esp_lab.diagnostics.two_reference_drift import validate_diagnostics

opened = {}
for key, path in outputs.items():
    dataset = xr.open_dataset(path)
    validate_diagnostics(dataset)
    assert 'Y' in dataset.dims and 'M' not in dataset.dims
    opened[key[0]] = dataset
for coordinate in ('Y', 'L', 'valid_time'):
    assert np.array_equal(opened['JRA55_FOSIRL'][coordinate], opened['Reanalysis'][coordinate])
{source: dict(dataset.sizes) for source, dataset in opened.items()}